In [ ]:
!pip install -q transformers torch scikit-learn pandas tqdm

In [ ]:
import pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import torch.nn.functional as F
from tqdm import tqdm
import os
import joblib
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random
from sklearn.utils import resample

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])

# Show first row to verify structure
print("Loaded dataset columns:", df.columns.tolist())

Loaded dataset columns: ['id', 'author', 'body', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'dominant']


In [ ]:
df['dominant'].value_counts()

,count
dominant,
neuroticism,4148
conscientiousness,1387
extraversion,1204
openness,683
agreeableness,620


In [ ]:
X_text = df["body"].tolist()
X_traits = df[["openness","conscientiousness","extraversion","agreeableness","neuroticism"]].values
y = LabelEncoder().fit_transform(df["dominant"])
scaler = StandardScaler(); X_traits = scaler.fit_transform(X_traits)
X_train_t, X_val_t, X_train_n, X_val_n, y_train, y_val = train_test_split(X_text, X_traits, y, test_size=0.2, random_state=42)

In [ ]:
class TraitDataset(Dataset):
    def __init__(self,texts,traits,labels,tokenizer,max_len=128):
        self.texts=texts;self.traits=torch.tensor(traits,dtype=torch.float32)
        self.labels=torch.tensor(labels,dtype=torch.long);self.tok=tokenizer;self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self,idx):
        t=self.tok(self.texts[idx],truncation=True,padding='max_length',max_length=self.max_len,return_tensors='pt')
        item={k:v.squeeze(0) for k,v in t.items()}
        item["traits"]=self.traits[idx];item["labels"]=self.labels[idx];return item

In [ ]:
def train_model(model_name,epochs=3,bs=16,lr=2e-5):
    tok=AutoTokenizer.from_pretrained(model_name)
    tr=TraitDataset(X_train_t,X_train_n,y_train,tok);vl=TraitDataset(X_val_t,X_val_n,y_val,tok)
    train_dl=DataLoader(tr,batch_size=bs,shuffle=True);val_dl=DataLoader(vl,batch_size=bs)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    base=AutoModel.from_pretrained(model_name)
    hidden=base.config.hidden_size
    trait_fc=nn.Linear(5,hidden);clf=nn.Linear(hidden*2,5)
    model=nn.ModuleList([base,trait_fc,clf]);opt=AdamW(model.parameters(),lr=lr);loss_fn=nn.CrossEntropyLoss()
    model.to(device)

    ckpt_dir = f"./checkpoints_{model_name.replace('/','_')}"
    os.makedirs(ckpt_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        for b in tqdm(train_dl,desc=f"{model_name} | Epoch {epoch+1}/{epochs}"):
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]
            t_emb=torch.relu(model[1](traits))
            logits=model[2](torch.cat([out,t_emb],dim=1))
            loss=loss_fn(logits,labels);opt.zero_grad();loss.backward();opt.step()

        torch.save({
            "epoch": epoch+1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": opt.state_dict(),
        }, f"{ckpt_dir}/epoch_{epoch+1}.pt")

    model.eval();preds=[];targs=[]
    with torch.no_grad():
        for b in val_dl:
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]
            t_emb=torch.relu(model[1](traits))
            logits=model[2](torch.cat([out,t_emb],dim=1))
            preds.extend(torch.argmax(logits,dim=1).cpu().numpy());targs.extend(labels.cpu().numpy())
    return accuracy_score(targs,preds), f1_score(targs,preds,average='macro'), model

In [ ]:
for m in ["bert-base-uncased","distilbert-base-uncased","roberta-base"]:
    acc,f1=train_model(m)
    print(f"{m} | acc={acc:.3f} | f1={f1:.3f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
roberta-base | Epoch 1/3: 100%|██████████| 403/403 [02:15<00:00,  2.98it/s]
roberta-base | Epoch 2/3: 100%|██████████| 403/403 [02:15<00:00,  2.98it/s]
roberta-base | Epoch 3/3: 100%|██████████| 403/403 [02:15<00:00,  2.98it/s]


roberta-base | acc=0.744 | f1=0.628


In [ ]:
def train_optimized_distilbert(epochs=5, bs=16, lr=5e-5, freeze_layers=True):
    model_name = "distilbert-base-uncased"
    tok = AutoTokenizer.from_pretrained(model_name)
    tr = TraitDataset(X_train_t, X_train_n, y_train, tok)
    vl = TraitDataset(X_val_t, X_val_n, y_val, tok)
    train_dl = DataLoader(tr, batch_size=bs, shuffle=True)
    val_dl = DataLoader(vl, batch_size=bs)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    base = AutoModel.from_pretrained(model_name)
    hidden = base.config.hidden_size
    trait_fc = nn.Linear(5, hidden)
    classifier = nn.Linear(hidden*3, 5)

    if freeze_layers:
        for name, param in base.named_parameters():
            if "transformer.layer" in name:
                layer_num = int(name.split("layer.")[1].split(".")[0])
                if layer_num < 2:
                    param.requires_grad = False

    model = nn.ModuleList([base, trait_fc, classifier]).to(device)

    class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    total_steps = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=0, num_training_steps=total_steps)

    for epoch in range(epochs):
        model.train()
        for b in tqdm(train_dl, desc=f"DistilBERT Optimized | Epoch {epoch+1}/{epochs}"):
            ids, mask, traits, labels = b['input_ids'].to(device), b['attention_mask'].to(device), b['traits'].to(device), b['labels'].to(device)
            out = model[0](input_ids=ids, attention_mask=mask).last_hidden_state
            cls = out[:,0,:]
            mean_pool = (out * mask.unsqueeze(-1)).sum(1) / mask.sum(1).unsqueeze(-1)
            text_vec = torch.cat([cls, mean_pool], dim=1)
            trait_vec = torch.relu(model[1](traits))
            logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
            loss = loss_fn(logits, labels)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()

        torch.save(model.state_dict(), f"distilbert_epoch{epoch+1}.pt")

    model.eval(); preds=[]; targs=[]
    with torch.no_grad():
        for b in val_dl:
            ids, mask, traits, labels = b['input_ids'].to(device), b['attention_mask'].to(device), b['traits'].to(device), b['labels'].to(device)
            out = model[0](input_ids=ids, attention_mask=mask).last_hidden_state
            cls = out[:,0,:]
            mean_pool = (out * mask.unsqueeze(-1)).sum(1) / mask.sum(1).unsqueeze(-1)
            text_vec = torch.cat([cls, mean_pool], dim=1)
            trait_vec = torch.relu(model[1](traits))
            logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy()); targs.extend(labels.cpu().numpy())

    return accuracy_score(targs, preds), f1_score(targs, preds, average='macro'), model

In [ ]:
acc, f1, best_model = train_optimized_distilbert(
    epochs=5, lr=5e-5, freeze_layers=True
)
print(f"Optimized DistilBERT | acc={acc:.3f} | f1={f1:.3f}")
torch.save(best_model.state_dict(), "dominant_classifier_distilbert_optimized.pt")

DistilBERT Optimized | Epoch 1/5: 100%|██████████| 403/403 [01:00<00:00,  6.64it/s]
DistilBERT Optimized | Epoch 2/5: 100%|██████████| 403/403 [01:00<00:00,  6.67it/s]
DistilBERT Optimized | Epoch 3/5: 100%|██████████| 403/403 [01:00<00:00,  6.66it/s]
DistilBERT Optimized | Epoch 4/5: 100%|██████████| 403/403 [01:00<00:00,  6.66it/s]
DistilBERT Optimized | Epoch 5/5: 100%|██████████| 403/403 [01:00<00:00,  6.65it/s]


Optimized DistilBERT | acc=0.743 | f1=0.643


# Test with some random data

In [ ]:
model_name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)
base = AutoModel.from_pretrained(model_name)
hidden = base.config.hidden_size

trait_fc = nn.Linear(5, hidden)
classifier = nn.Linear(hidden*3, 5)
model = nn.ModuleList([base, trait_fc, classifier])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.load_state_dict(torch.load("dominant_classifier_distilbert_optimized.pt", map_location=device))
model.to(device)
model.eval()

labels = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']

def predict_dominant(body, traits):
    t = tok(body, truncation=True, padding='max_length', max_length=160, return_tensors='pt').to(device)
    traits = torch.tensor(traits, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model[0](**t).last_hidden_state
        cls = out[:,0,:]
        mean_pool = (out * t['attention_mask'].unsqueeze(-1)).sum(1) / t['attention_mask'].sum(1).unsqueeze(-1)
        text_vec = torch.cat([cls, mean_pool], dim=1)
        trait_vec = torch.relu(model[1](traits))
        logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
        probs = F.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = torch.argmax(logits, dim=1).item()

    result = {labels[i]: round(float(probs[i]), 3) for i in range(len(labels))}
    return labels[pred], result

In [ ]:
samples = [
    ("I often skip meals when I’m stressed or anxious.", [0.45, 0.38, 0.42, 0.51, 0.77]),
    ("I love exploring new cuisines and trying creative recipes.", [0.81, 0.62, 0.69, 0.64, 0.35]),
    ("I plan my meals ahead of time and stick to my schedule.", [0.48, 0.83, 0.52, 0.61, 0.29]),
    ("I get distracted easily while eating or multitasking.", [0.37, 0.41, 0.56, 0.44, 0.68])
]

for text, traits in samples:
    pred, scores = predict_dominant(text, traits)
    print(f"\nText: {text}\nPredicted Dominant Trait: {pred}\nConfidence Scores: {scores}")


Text: I often skip meals when I’m stressed or anxious.
Predicted Dominant Trait: agreeableness
Confidence Scores: {'openness': 0.0, 'conscientiousness': 0.0, 'extraversion': 0.0, 'agreeableness': 0.999, 'neuroticism': 0.0}

Text: I love exploring new cuisines and trying creative recipes.
Predicted Dominant Trait: conscientiousness
Confidence Scores: {'openness': 0.002, 'conscientiousness': 0.623, 'extraversion': 0.3, 'agreeableness': 0.034, 'neuroticism': 0.041}

Text: I plan my meals ahead of time and stick to my schedule.
Predicted Dominant Trait: conscientiousness
Confidence Scores: {'openness': 0.011, 'conscientiousness': 0.944, 'extraversion': 0.037, 'agreeableness': 0.006, 'neuroticism': 0.003}

Text: I get distracted easily while eating or multitasking.
Predicted Dominant Trait: agreeableness
Confidence Scores: {'openness': 0.001, 'conscientiousness': 0.002, 'extraversion': 0.009, 'agreeableness': 0.988, 'neuroticism': 0.001}


In [ ]:
# load the same dataset
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])

# pick 5 random rows
samples = df.sample(10, random_state=42)

for _, row in samples.iterrows():
    body = row["body"]
    traits = [row["openness"], row["conscientiousness"], row["extraversion"],
              row["agreeableness"], row["neuroticism"]]
    true_label = row["dominant"]

    pred_label, conf = predict_dominant(body, traits)
    print(f"\nText: {body[:120]}{'...' if len(body)>120 else ''}")
    print(f"True Label: {true_label}")
    print(f"Predicted Label: {pred_label}")
    print(f"Confidence Scores: {conf}")


Text: i suffer with an eating disorder and my mum expects it to go overnight removed
True Label: neuroticism
Predicted Label: agreeableness
Confidence Scores: {'openness': 0.0, 'conscientiousness': 0.0, 'extraversion': 0.003, 'agreeableness': 0.996, 'neuroticism': 0.0}

Text: homemade corned beef and luncheon meat topped with noodles
True Label: agreeableness
Predicted Label: agreeableness
Confidence Scores: {'openness': 0.067, 'conscientiousness': 0.126, 'extraversion': 0.062, 'agreeableness': 0.711, 'neuroticism': 0.033}

Text: i want to get worse and i hate it i used to be a prominent user on edtwt with about followers not a huge amount but it s...
True Label: neuroticism
Predicted Label: agreeableness
Confidence Scores: {'openness': 0.0, 'conscientiousness': 0.003, 'extraversion': 0.005, 'agreeableness': 0.992, 'neuroticism': 0.001}

Text: hey do you think i should tell my therapist i told her im not hungry but i just dont want to eat sometimes i dont eat at...
True Label: neuroti

In [ ]:
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
minor = df[df['dominant']=='agreeableness']
major = df[df['dominant']!='agreeableness']
minor_up = resample(minor, replace=True, n_samples=len(major), random_state=42)
df = pd.concat([major, minor_up])

# ---------- 2. Prepare features ----------
X_text = df["body"].tolist()
X_traits = df[["openness","conscientiousness","extraversion","agreeableness","neuroticism"]].values
y = LabelEncoder().fit_transform(df["dominant"])
scaler = StandardScaler(); X_traits = scaler.fit_transform(X_traits)
X_train_t, X_val_t, X_train_n, X_val_n, y_train, y_val = train_test_split(X_text, X_traits, y, test_size=0.2, random_state=42)

# ---------- 3. Dataset ----------
class TraitDataset(Dataset):
    def __init__(self,texts,traits,labels,tokenizer,max_len=160):
        self.texts=texts;self.traits=torch.tensor(traits,dtype=torch.float32)
        self.labels=torch.tensor(labels,dtype=torch.long);self.tok=tokenizer;self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self,idx):
        t=self.tok(self.texts[idx],truncation=True,padding='max_length',max_length=self.max_len,return_tensors='pt')
        item={k:v.squeeze(0) for k,v in t.items()}
        item["traits"]=self.traits[idx];item["labels"]=self.labels[idx];return item

# ---------- 4. Train ----------
def train_emotion_model(epochs=4,bs=16,lr=5e-5):
    model_name = "j-hartmann/emotion-english-distilroberta-base"
    tok = AutoTokenizer.from_pretrained(model_name)
    tr = TraitDataset(X_train_t,X_train_n,y_train,tok); vl = TraitDataset(X_val_t,X_val_n,y_val,tok)
    train_dl = DataLoader(tr,batch_size=bs,shuffle=True); val_dl = DataLoader(vl,batch_size=bs)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    base=AutoModel.from_pretrained(model_name)
    hidden=base.config.hidden_size
    trait_fc=nn.Linear(5,hidden)
    classifier=nn.Linear(hidden*3,5)

    # Freeze first two transformer layers
    for name, param in base.named_parameters():
        if "encoder.layer" in name and any(str(i) in name for i in range(0,2)):
            param.requires_grad=False

    model=nn.ModuleList([base,trait_fc,classifier]).to(device)
    loss_fn=nn.CrossEntropyLoss()
    opt=AdamW(filter(lambda p:p.requires_grad, model.parameters()),lr=lr)

    for epoch in range(epochs):
        model.train()
        for b in tqdm(train_dl,desc=f"Emotion-RoBERTa Epoch {epoch+1}/{epochs}"):
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state
            cls=out[:,0,:]
            mean_pool=(out*mask.unsqueeze(-1)).sum(1)/mask.sum(1).unsqueeze(-1)
            text_vec=torch.cat([cls,mean_pool],dim=1)
            trait_vec=torch.relu(model[1](traits))
            logits=model[2](torch.cat([text_vec,trait_vec],dim=1))
            loss=loss_fn(logits,labels)
            opt.zero_grad();loss.backward();opt.step()

        torch.save(model.state_dict(),f"emotion_roberta_epoch{epoch+1}.pt")

    model.eval();preds=[];targs=[]
    with torch.no_grad():
        for b in val_dl:
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state
            cls=out[:,0,:]
            mean_pool=(out*mask.unsqueeze(-1)).sum(1)/mask.sum(1).unsqueeze(-1)
            text_vec=torch.cat([cls,mean_pool],dim=1)
            trait_vec=torch.relu(model[1](traits))
            logits=model[2](torch.cat([text_vec,trait_vec],dim=1))
            preds.extend(torch.argmax(logits,dim=1).cpu().numpy());targs.extend(labels.cpu().numpy())

    return accuracy_score(targs,preds), f1_score(targs,preds,average='macro'), model

In [ ]:
acc, f1, best_model = train_emotion_model()
print(f"Emotion-RoBERTa | acc={acc:.3f} | f1={f1:.3f}")
torch.save(best_model.state_dict(), "dominant_classifier_emotion_roberta.pt")

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Emotion-RoBERTa Epoch 1/4:   0%|          | 2/743 [00:00<01:50,  6.73it/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Emotion-RoBERTa Epoch 4/4: 100%|██████████| 743/743 [02:20<00:00,  5.30it/s]


Emotion-RoBERTa | acc=0.871 | f1=0.750


In [ ]:
# ---- load model ----
model_name = "j-hartmann/emotion-english-distilroberta-base"
tok = AutoTokenizer.from_pretrained(model_name)
base = AutoModel.from_pretrained(model_name)
hidden = base.config.hidden_size
trait_fc = nn.Linear(5, hidden)
classifier = nn.Linear(hidden*3, 5)
model = nn.ModuleList([base, trait_fc, classifier])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.load_state_dict(torch.load("dominant_classifier_emotion_roberta.pt", map_location=device))
model.to(device)
model.eval()

labels = ['openness','conscientiousness','extraversion','agreeableness','neuroticism']

# ---- prediction ----
def predict_emotion_roberta(body, traits):
    t = tok(body, truncation=True, padding='max_length', max_length=160, return_tensors='pt').to(device)
    traits = torch.tensor(traits, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model[0](**t).last_hidden_state
        cls = out[:,0,:]
        mean_pool = (out * t['attention_mask'].unsqueeze(-1)).sum(1) / t['attention_mask'].sum(1).unsqueeze(-1)
        text_vec = torch.cat([cls, mean_pool], dim=1)
        trait_vec = torch.relu(model[1](traits))
        logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
        probs = F.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = torch.argmax(logits, dim=1).item()
    return labels[pred], {labels[i]: round(float(probs[i]),3) for i in range(len(labels))}

Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
samples = df.sample(5, random_state=35)

for _, row in samples.iterrows():
    text = row["body"]
    traits = [row["openness"], row["conscientiousness"], row["extraversion"],
              row["agreeableness"], row["neuroticism"]]
    true = row["dominant"]
    pred, conf = predict_emotion_roberta(text, traits)
    print(f"\nText: {text[:120]}{'...' if len(text)>120 else ''}")
    print(f"True: {true} | Pred: {pred}")
    print(f"Confidence: {conf}")


Text: getting covid made me worse too i understand about you liking the feeling of hunger and not knowing what to do if you wa...
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.0, 'conscientiousness': 0.0, 'extraversion': 0.001, 'agreeableness': 0.997, 'neuroticism': 0.002}

Text: but it isn t raw because smoking is a form of cooking that s what he was getting at
True: conscientiousness | Pred: conscientiousness
Confidence: {'openness': 0.002, 'conscientiousness': 0.889, 'extraversion': 0.039, 'agreeableness': 0.04, 'neuroticism': 0.029}

Text: can anyone relate did anyone else turn to self harm after stopping restricting i m a yo f and stopped restricting a few ...
True: conscientiousness | Pred: conscientiousness
Confidence: {'openness': 0.002, 'conscientiousness': 0.601, 'extraversion': 0.065, 'agreeableness': 0.31, 'neuroticism': 0.021}

Text: pumpkin goat cheese baked gnocchi removed
True: conscientiousness | Pred: agreeableness
Confidence: {'openness': 0.006,

# More fine-tuning

In [ ]:
# ---------- 1. Re-balance (boost neuroticism) ----------
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
minor = df[df['dominant']=='agreeableness']
major = df[df['dominant']!='agreeableness']
minor_up = resample(minor, replace=True, n_samples=len(major), random_state=42)
df = pd.concat([major, minor_up])

# extra neuroticism boost
df_neuro = df[df['dominant']=='neuroticism']
df = pd.concat([df, df_neuro])

X_text = df["body"].tolist()
X_traits = df[["openness","conscientiousness","extraversion","agreeableness","neuroticism"]].values
y = LabelEncoder().fit_transform(df["dominant"])
scaler = StandardScaler(); X_traits = scaler.fit_transform(X_traits)
X_train_t, X_val_t, X_train_n, X_val_n, y_train, y_val = train_test_split(X_text, X_traits, y, test_size=0.2, random_state=42)

# ---------- 2. Dataset ----------
class TraitDataset(Dataset):
    def __init__(self,texts,traits,labels,tokenizer,max_len=160):
        self.texts=texts;self.traits=torch.tensor(traits,dtype=torch.float32)
        self.labels=torch.tensor(labels,dtype=torch.long);self.tok=tokenizer;self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self,idx):
        t=self.tok(self.texts[idx],truncation=True,padding='max_length',max_length=self.max_len,return_tensors='pt')
        item={k:v.squeeze(0) for k,v in t.items()}
        item["traits"]=self.traits[idx];item["labels"]=self.labels[idx];return item

# ---------- 3. Focal Loss + label-smoothing ----------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smooth=0.1):
        super().__init__()
        self.gamma=gamma; self.smooth=smooth
    def forward(self, logits, targets):
        num_classes = logits.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smooth / (num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1 - self.smooth)
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)
        loss = -( (1 - probs) ** self.gamma * log_probs * true_dist ).sum(dim=1).mean()
        return loss

# ---------- 4. Train ----------
def train_emotion_booster(epochs=2,bs=16,lr=4e-5):
    model_name = "j-hartmann/emotion-english-distilroberta-base"
    tok = AutoTokenizer.from_pretrained(model_name)
    tr = TraitDataset(X_train_t,X_train_n,y_train,tok); vl = TraitDataset(X_val_t,X_val_n,y_val,tok)
    train_dl = DataLoader(tr,batch_size=bs,shuffle=True); val_dl = DataLoader(vl,batch_size=bs)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    base=AutoModel.from_pretrained(model_name)
    hidden=base.config.hidden_size
    trait_fc=nn.Linear(5,hidden)
    classifier=nn.Linear(hidden*3,5)

    model=nn.ModuleList([base,trait_fc,classifier]).to(device)
    loss_fn=FocalLoss(gamma=2.0,smooth=0.1)
    opt=AdamW(model.parameters(),lr=lr)

    for epoch in range(epochs):
        model.train()
        for b in tqdm(train_dl,desc=f"Booster Epoch {epoch+1}/{epochs}"):
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state
            cls=out[:,0,:]
            mean_pool=(out*mask.unsqueeze(-1)).sum(1)/mask.sum(1).unsqueeze(-1)
            text_vec=torch.cat([cls,mean_pool],dim=1)
            trait_vec=torch.relu(model[1](traits))
            logits=model[2](torch.cat([text_vec,trait_vec],dim=1))
            loss=loss_fn(logits,labels)
            opt.zero_grad();loss.backward();opt.step()
        torch.save(model.state_dict(),f"emotion_roberta_booster_epoch{epoch+1}.pt")

    model.eval();preds=[];targs=[]
    with torch.no_grad():
        for b in val_dl:
            ids,mask,traits,labels=b['input_ids'].to(device),b['attention_mask'].to(device),b['traits'].to(device),b['labels'].to(device)
            out=model[0](input_ids=ids,attention_mask=mask).last_hidden_state
            cls=out[:,0,:]
            mean_pool=(out*mask.unsqueeze(-1)).sum(1)/mask.sum(1).unsqueeze(-1)
            text_vec=torch.cat([cls,mean_pool],dim=1)
            trait_vec=torch.relu(model[1](traits))
            logits=model[2](torch.cat([text_vec,trait_vec],dim=1))
            preds.extend(torch.argmax(logits,dim=1).cpu().numpy());targs.extend(labels.cpu().numpy())
    return accuracy_score(targs,preds), f1_score(targs,preds,average='macro'), model

In [ ]:
acc, f1, booster_model = train_emotion_booster()
print(f"Booster Fine-Tune | acc={acc:.3f} | f1={f1:.3f}")
torch.save(booster_model.state_dict(), "booster_dominant_classifier_emotion_roberta.pt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Booster Epoch 1/2:   0%|          | 0/950 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Booster Epoch 2/2: 100%|██████████| 950/950 [03:22<00:00,  4.69it/s]


Booster Fine-Tune | acc=0.919 | f1=0.773


In [ ]:
# ---- load model ----
model_name = "j-hartmann/emotion-english-distilroberta-base"
tok = AutoTokenizer.from_pretrained(model_name)
base = AutoModel.from_pretrained(model_name)
hidden = base.config.hidden_size
trait_fc = nn.Linear(5, hidden)
classifier = nn.Linear(hidden*3, 5)
model = nn.ModuleList([base, trait_fc, classifier])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.load_state_dict(torch.load("booster_dominant_classifier_emotion_roberta.pt", map_location=device))
model.to(device)
model.eval()

labels = ['openness','conscientiousness','extraversion','agreeableness','neuroticism']

# ---- prediction ----
def predict_emotion_roberta_boosted(body, traits):
    t = tok(body, truncation=True, padding='max_length', max_length=160, return_tensors='pt').to(device)
    traits = torch.tensor(traits, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model[0](**t).last_hidden_state
        cls = out[:,0,:]
        mean_pool = (out * t['attention_mask'].unsqueeze(-1)).sum(1) / t['attention_mask'].sum(1).unsqueeze(-1)
        text_vec = torch.cat([cls, mean_pool], dim=1)
        trait_vec = torch.relu(model[1](traits))
        logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
        probs = F.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = torch.argmax(logits, dim=1).item()
    return labels[pred], {labels[i]: round(float(probs[i]),3) for i in range(len(labels))}

Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
samples = df.sample(5, random_state=65)

for _, row in samples.iterrows():
    text = row["body"]
    traits = [row["openness"], row["conscientiousness"], row["extraversion"],
              row["agreeableness"], row["neuroticism"]]
    true = row["dominant"]
    pred, conf = predict_emotion_roberta_boosted(text, traits)
    print(f"\nText: {text[:120]}{'...' if len(text)>120 else ''}")
    print(f"True: {true} | Pred: {pred}")
    print(f"Confidence: {conf}")


Text: how do you recover from an ed if you need to lose weight we can all agree that you can have an ed at any shape or size i...
True: extraversion | Pred: agreeableness
Confidence: {'openness': 0.087, 'conscientiousness': 0.152, 'extraversion': 0.239, 'agreeableness': 0.43, 'neuroticism': 0.091}

Text: asmr oreo party cookies strawberry cream puff strawberry clair oreo crepe oreo cookies
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.05, 'conscientiousness': 0.084, 'extraversion': 0.093, 'agreeableness': 0.704, 'neuroticism': 0.068}

Text: does anyone else struggle with imposter syndrome while recovering and if so have you found anything that helps i m only ...
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.101, 'conscientiousness': 0.092, 'extraversion': 0.095, 'agreeableness': 0.625, 'neuroticism': 0.087}

Text: minute healthy breakfast recipe deleted
True: agreeableness | Pred: openness
Confidence: {'openness': 0.56, 'conscientiousness': 0.

#Ensemble

In [ ]:
# --- shared config ---
model_name = "j-hartmann/emotion-english-distilroberta-base"
labels = ['openness','conscientiousness','extraversion','agreeableness','neuroticism']
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- base structure factory ---
def build_model():
    base = AutoModel.from_pretrained(model_name)
    hidden = base.config.hidden_size
    trait_fc = nn.Linear(5, hidden)
    classifier = nn.Linear(hidden*3, 5)
    model = nn.ModuleList([base, trait_fc, classifier]).to(device)
    return model

# --- load both checkpoints ---
model_main = build_model()
model_main.load_state_dict(torch.load("dominant_classifier_emotion_roberta.pt", map_location=device))
model_main.eval()

model_boost = build_model()
model_boost.load_state_dict(torch.load("booster_dominant_classifier_emotion_roberta.pt", map_location=device))
model_boost.eval()

tok = AutoTokenizer.from_pretrained(model_name)

# --- inference with weighted ensemble ---
def predict_ensemble(body, traits, w_main=0.7, w_boost=0.3):
    t = tok(body, truncation=True, padding='max_length', max_length=160, return_tensors='pt').to(device)
    traits = torch.tensor(traits, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        def forward_pass(model):
            out = model[0](**t).last_hidden_state
            cls = out[:,0,:]
            mean_pool = (out * t['attention_mask'].unsqueeze(-1)).sum(1) / t['attention_mask'].sum(1).unsqueeze(-1)
            text_vec = torch.cat([cls, mean_pool], dim=1)
            trait_vec = torch.relu(model[1](traits))
            logits = model[2](torch.cat([text_vec, trait_vec], dim=1))
            return logits

        logits_main = forward_pass(model_main)
        logits_boost = forward_pass(model_boost)

        logits = w_main*logits_main + w_boost*logits_boost
        probs = F.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = np.argmax(probs)
    return labels[pred], {labels[i]: round(float(probs[i]),3) for i in range(len(labels))}

Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at j-hartmann/emotion-english-distilroberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
samples = df.sample(5, random_state=16)

for _, row in samples.iterrows():
    text = row["body"]
    traits = [row["openness"], row["conscientiousness"], row["extraversion"],
              row["agreeableness"], row["neuroticism"]]
    true = row["dominant"]
    pred, conf = predict_ensemble(text, traits)
    print(f"\nText: {text[:120]}{'...' if len(text)>120 else ''}")
    print(f"True: {true} | Pred: {pred}")
    print(f"Confidence: {conf}")


Text: i m personally not a fan of seafood but i d eat that steak and potato
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.001, 'conscientiousness': 0.02, 'extraversion': 0.046, 'agreeableness': 0.925, 'neuroticism': 0.008}

Text: it s not necessarily ed but it could turn into one you might be like a lot of people where you don t feel hungry but the...
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.001, 'conscientiousness': 0.006, 'extraversion': 0.03, 'agreeableness': 0.952, 'neuroticism': 0.011}

Text: i ate trahan traditional albanian breakfast ground wheat and milk
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.005, 'conscientiousness': 0.005, 'extraversion': 0.003, 'agreeableness': 0.987, 'neuroticism': 0.001}

Text: i ate a krispy kreme cheeseburger
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.001, 'conscientiousness': 0.015, 'extraversion': 0.031, 'agreeableness': 0.906, 'neuroticism': 0.047}

Tex

#Sentence Transformer + MLP

In [ ]:
!pip install -q sentence-transformers torch scikit-learn pandas tqdm

In [ ]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# ---------- 1. Load & prepare data ----------
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
texts = df["body"].tolist()
traits = df[["openness","conscientiousness","extraversion","agreeableness","neuroticism"]].values
y = LabelEncoder().fit_transform(df["dominant"])
scaler = StandardScaler(); traits = scaler.fit_transform(traits)

X_train_t, X_test_t, X_train_traits, X_test_traits, y_train, y_test = train_test_split(
    texts, traits, y, test_size=0.2, random_state=42
)

# ---------- 2. Text embeddings ----------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")  # 384-dim
X_train_text_emb = embedder.encode(X_train_t, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
X_test_text_emb  = embedder.encode(X_test_t, show_progress_bar=True, batch_size=64, convert_to_numpy=True)

# Concatenate traits → final feature = 384 + 5
X_train = np.hstack((X_train_text_emb, X_train_traits))
X_test  = np.hstack((X_test_text_emb,  X_test_traits))

# ---------- 3. PyTorch dataset ----------
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_ds  = TensorDataset(torch.tensor(X_test, dtype=torch.float32),  torch.tensor(y_test,  dtype=torch.long))
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=32)

# ---------- 4. MLP model ----------
class MLPClassifier(nn.Module):
    def __init__(self, in_dim=389, hidden=256, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPClassifier(in_dim=X_train.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

# ---------- 5. Train ----------
for epoch in range(6):
    model.train(); total, correct, running_loss = 0, 0, 0
    for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch+1}/6"):
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        running_loss += loss.item() * xb.size(0)
        correct += (pred.argmax(1) == yb).sum().item()
        total += yb.size(0)
    train_acc = correct/total; train_loss = running_loss/total
    print(f"Train Acc: {train_acc:.3f}, Loss: {train_loss:.4f}")

# ---------- 6. Evaluate ----------
def evaluate(dataloader):
    model.eval(); preds, targs = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())
            targs.extend(yb.cpu().numpy())
    return accuracy_score(targs, preds), f1_score(targs, preds, average="macro")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

Train Acc: 0.828, Loss: 0.4431


Epoch 5/6: 100%|██████████| 202/202 [00:00<00:00, 419.31it/s]


Train Acc: 0.838, Loss: 0.4018


Epoch 6/6: 100%|██████████| 202/202 [00:00<00:00, 464.75it/s]

Train Acc: 0.846, Loss: 0.3819


In [ ]:
train_acc, train_f1 = evaluate(train_dl)
test_acc,  test_f1  = evaluate(test_dl)

print(f"\nMLP Train Acc: {train_acc:.3f} | Train F1: {train_f1:.3f}")
print(f"MLP Test  Acc: {test_acc:.3f} | Test  F1: {test_f1:.3f}")


✅ MLP Train Acc: 0.901 | Train F1: 0.865
✅ MLP Test  Acc: 0.849 | Test  F1: 0.800


In [ ]:
torch.save(model.state_dict(), "dominant_classifier_sentence_mlp.pt")

In [ ]:
joblib.dump(scaler, "trait_scaler.pkl")

['trait_scaler.pkl']

In [ ]:
# ---- Load model & components ----
class MLPClassifier(nn.Module):
    def __init__(self, in_dim=389, hidden=256, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPClassifier(in_dim=389).to(device)
model.load_state_dict(torch.load("dominant_classifier_sentence_mlp.pt", map_location=device))
model.eval()

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")
scaler = joblib.load("trait_scaler.pkl")

labels = ['openness','conscientiousness','extraversion','agreeableness','neuroticism']

# ---- Prediction function ----
def predict_mlp(body, traits):
    traits_scaled = scaler.transform([traits])
    text_emb = embedder.encode([body], convert_to_numpy=True)
    X = np.hstack((text_emb, traits_scaled))
    X = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X)
        probs = F.softmax(logits, dim=1).cpu().numpy().flatten()
        pred = np.argmax(probs)
    return labels[pred], {labels[i]: round(float(probs[i]),3) for i in range(len(labels))}

In [ ]:
# ---- Random dataset test ----
df = pd.read_csv("combined_cleaned_dataset.csv").dropna(subset=["body","dominant"])
samples = df.sample(5, random_state=95)

for _, row in samples.iterrows():
    text = row["body"]
    traits = [row["openness"], row["conscientiousness"], row["extraversion"],
              row["agreeableness"], row["neuroticism"]]
    true = row["dominant"]
    pred, conf = predict_mlp(text, traits)
    print(f"\nText: {text[:120]}{'...' if len(text)>120 else ''}")
    print(f"True: {true} | Pred: {pred}")
    print(f"Confidence: {conf}")


Text: the only tips i can give you besides seeking professional help is to be kind to yourself a body cannot survive on that a...
True: conscientiousness | Pred: conscientiousness
Confidence: {'openness': 0.026, 'conscientiousness': 0.936, 'extraversion': 0.0, 'agreeableness': 0.037, 'neuroticism': 0.0}

Text: i m tired of people praising my body i know it sounds really conceited in the title but please listen i ve suffered from...
True: extraversion | Pred: extraversion
Confidence: {'openness': 0.053, 'conscientiousness': 0.0, 'extraversion': 0.918, 'agreeableness': 0.029, 'neuroticism': 0.0}

Text: your post has been removed this subreddit is to encourage and help one another seek help for ed and celebrate getting he...
True: neuroticism | Pred: agreeableness
Confidence: {'openness': 0.0, 'conscientiousness': 0.0, 'extraversion': 0.0, 'agreeableness': 1.0, 'neuroticism': 0.0}

Text: you could gather insight about what an eating disorder is and how it usually is treated but honestly 